In [1]:
!pip install -q torch torchaudio scikit-learn matplotlib tqdm pyyaml

In [2]:
import zipfile, os
# If you uploaded B22AI058.zip:
with zipfile.ZipFile("B22AI058.zip") as z:
    z.extractall(".")
os.chdir("q2")

In [6]:
os.makedirs("./data", exist_ok=True)
os.makedirs("exps/baseline", exist_ok=True)
os.makedirs("exps/disentangler", exist_ok=True)
os.makedirs("exps/improved", exist_ok=True)

In [7]:
!python train.py --config configs/baseline.yaml

2026-03-27 13:48:34,440 INFO Using device: cuda | Mode: baseline
100% 5.95G/5.95G [05:24<00:00, 19.7MB/s]
2026-03-27 13:57:44,623 INFO Baseline dataset: 28539 utts, 251 speakers
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Baseline Epoch 1:   0% 0/445 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive

In [8]:
!python train.py --config configs/disentangler.yaml


2026-03-27 16:13:59,428 INFO Using device: cuda | Mode: disentangler
2026-03-27 16:13:59,740 INFO Loading LibriSpeech train-clean-100 from ./data...
2026-03-27 16:16:19,174 INFO Dataset: 2090 triplets, 209 speakers
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[disentangler] Epoch 1:   0% 0/65 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going

In [9]:
!python train.py --config configs/improved.yaml


2026-03-27 16:46:48,671 INFO Using device: cuda | Mode: improved
2026-03-27 16:46:48,888 INFO Loading LibriSpeech train-clean-100 from ./data...
2026-03-27 16:49:06,291 INFO Dataset: 2090 triplets, 209 speakers
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/q2/train.py", line 702, in <module>
    main()
  File "/content/q2/train.py", line 696, in main
    train_disentangler(cfg, device, mode=cfg["mode"])
  File "/content/q2/train.py", line 507, in train_disentangler
    cfg["aam_margin"], cfg["aam_scale"]).to(d

In [10]:
!python eval.py \
  --config configs/disentangler.yaml \
  --checkpoints \
    exps/baseline/ckpt_epoch015.pt \
    exps/disentangler/ckpt_epoch015.pt \
    exps/improved/ckpt_epoch015.pt \
  --names baseline disentangler improved \
  --results_dir results/

2026-03-27 16:49:12,375 INFO Evaluating baseline from exps/baseline/ckpt_epoch015.pt...
2026-03-27 16:49:12,376 INFO Running synthetic evaluation on LibriSpeech test-clean...
100% 331M/331M [00:19<00:00, 18.2MB/s]
2026-03-27 16:49:31,777 INFO ./data/LibriSpeech/LICENSE.TXT already extracted.
2026-03-27 16:49:31,777 INFO ./data/LibriSpeech/README.TXT already extracted.
2026-03-27 16:49:31,778 INFO ./data/LibriSpeech/CHAPTERS.TXT already extracted.
2026-03-27 16:49:31,781 INFO ./data/LibriSpeech/SPEAKERS.TXT already extracted.
2026-03-27 16:49:31,782 INFO ./data/LibriSpeech/BOOKS.TXT already extracted.
Traceback (most recent call last):
  File "/content/q2/eval.py", line 420, in <module>
    main()
  File "/content/q2/eval.py", line 381, in main
    eer, minDCF, scores, labels, embs, emb_labels = synthetic_eval(
                                                    ^^^^^^^^^^^^^^^
  File "/content/q2/eval.py", line 254, in synthetic_eval
    ckpt = torch.load(checkpoint_path, map_location=

In [11]:
import pandas as pd
pd.read_csv("results/tables/results.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'results/tables/results.csv'

In [ ]:
from IPython.display import Image
Image("results/plots/det_all_systems.png")

In [ ]:
import shutil
from google.colab import files

# Zip the entire q2 folder
shutil.make_archive("/content/q2_results", "zip", "/content", "q2")

# Download it
files.download("/content/q2_results.zip")